# CineScope — Core Analytics (§2.3)

**Local Spark only** (`local[*]`). Reads silver `movies_awards_enriched`.
Charts are **shown inline** and also written under `outputs/charts/generated/`.

Insights:
1. Genre × decade median rating / votes  
2. Director prior track record vs rating  
3. Runtime sweet spot  
4. High-rating / low-vote anomalies  
5. Pre- vs post-release signal

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from dotenv import load_dotenv

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from cinescope.analytics import (
    director_prior_correlation,
    director_prior_vs_rating,
    genre_decade_stats,
    rating_anomalies,
    runtime_bucket_stats,
)
from cinescope.ml.features import with_hit_label
from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=True, validate_mount=True)
spark = build_spark_session(app_name="cinescope-core-analytics", paths=paths)
spark.sparkContext.setLogLevel("WARN")

charts_dir = REPO / "outputs" / "charts" / "generated"
metrics_dir = REPO / "outputs" / "metrics"
charts_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

def save_and_show(fig, path: Path):
    """Write PNG and display inline in the notebook."""
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", path)

movies = spark.read.parquet(str(paths.movies_awards_enriched_dir))
n = movies.count()
print("rows:", n)
print("path:", paths.movies_awards_enriched_dir)
movies.printSchema()

## 1. Genre × decade

In [ ]:
gdec = genre_decade_stats(movies)
gdec_pd = gdec.filter(F.col("film_count") >= 50).toPandas()
top_genres = (
    gdec_pd.groupby("genre")["film_count"].sum().sort_values(ascending=False).head(8).index.tolist()
)
plot_df = gdec_pd[gdec_pd["genre"].isin(top_genres)]

fig, ax = plt.subplots(figsize=(10, 5))
for genre, part in plot_df.groupby("genre"):
    part = part.sort_values("decade")
    ax.plot(part["decade"], part["median_rating"], marker="o", label=genre)
ax.set_xlabel("Decade")
ax.set_ylabel("Median IMDb rating")
ax.set_title("Median rating by genre and decade (n≥50 per cell)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
p1 = charts_dir / "genre_decade_median_rating.png"
save_and_show(fig, p1)
gdec_pd.head(12)

## 2. Director prior vs rating

In [ ]:
corr = director_prior_correlation(movies)
dir_pd = director_prior_vs_rating(movies).sample(False, 0.05, seed=42).toPandas()
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(dir_pd["director_prior_rating_mean"], dir_pd["average_rating"], s=8, alpha=0.25)
ax.set_xlabel("Director prior mean rating (leakage-safe)")
ax.set_ylabel("Film average rating")
ax.set_title(f"Director track record vs film rating (corr={corr:.3f}, 5% sample)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
p2 = charts_dir / "director_prior_vs_rating.png"
save_and_show(fig, p2)
print("corr", corr)

## 3. Runtime sweet spot

In [ ]:
rt = runtime_bucket_stats(movies).toPandas()
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(rt["runtime_bucket"], rt["film_count"], width=12, alpha=0.35, label="film count")
ax2 = ax1.twinx()
ax2.plot(rt["runtime_bucket"], rt["median_rating"], color="C3", marker="o", label="median rating")
ax1.set_xlabel("Runtime bucket (minutes)")
ax1.set_ylabel("Films")
ax2.set_ylabel("Median rating")
ax1.set_title("Runtime sweet spot: volume vs median rating")
fig.tight_layout()
p3 = charts_dir / "runtime_sweet_spot.png"
save_and_show(fig, p3)
rt.sort_values("median_rating", ascending=False).head(3)

## 4. Rating anomalies (high rating, low votes)

In [ ]:
anom = rating_anomalies(movies, high_rating=8.0, vote_percentile=0.10)
anom_n = anom.count()
anom_pd = anom.limit(5000).toPandas()
sample_all = movies.select("average_rating", "num_votes").sample(False, 0.02, seed=7).toPandas()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample_all["num_votes"].clip(lower=1), sample_all["average_rating"], s=6, alpha=0.15, label="sample")
ax.scatter(anom_pd["num_votes"].clip(lower=1), anom_pd["average_rating"], s=10, alpha=0.5, label="anomalies")
ax.set_xscale("log")
ax.set_xlabel("num_votes (log)")
ax.set_ylabel("average_rating")
ax.set_title(f"High rating (≥8) with bottom-decile votes (n={anom_n})")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
p4 = charts_dir / "rating_vote_anomalies.png"
save_and_show(fig, p4)
anom.limit(10).toPandas()

## 5. Pre- vs post-release signal

Note: hit label requires `num_votes >= 1000`, so high vote quartiles are partly mechanical.

In [ ]:
labeled = with_hit_label(movies)
hit_rate = labeled.agg(F.avg("is_hit")).first()[0]

pre = (
    labeled.groupBy("has_known_director")
    .agg(F.count("*").alias("n"), F.avg("is_hit").alias("hit_rate"))
    .orderBy("has_known_director")
    .toPandas()
)
post = (
    labeled.withColumn("vote_quartile", F.ntile(4).over(Window.orderBy("num_votes")))
    .groupBy("vote_quartile")
    .agg(F.count("*").alias("n"), F.avg("is_hit").alias("hit_rate"), F.avg("num_votes").alias("avg_votes"))
    .orderBy("vote_quartile")
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(pre["has_known_director"].astype(str), pre["hit_rate"])
axes[0].set_title("Hit rate by has_known_director (pre-release)")
axes[0].set_ylabel("P(is_hit)")
axes[1].bar(post["vote_quartile"].astype(str), post["hit_rate"])
axes[1].set_title("Hit rate by vote quartile (post-release)")
axes[1].set_xlabel("vote_quartile (1=low)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
p5 = charts_dir / "pre_vs_post_release_hit_rate.png"
save_and_show(fig, p5)
print("base hit rate", hit_rate)
print(pre)
print(post)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

metrics = {
    "job": "core_analytics",
    "input": str(paths.movies_awards_enriched_dir),
    "row_count": n,
    "director_prior_rating_corr": corr,
    "anomaly_count_rating_ge_8_bottom_decile_votes": anom_n,
    "base_hit_rate": float(hit_rate) if hit_rate is not None else None,
    "charts": [str(p) for p in [p1, p2, p3, p4, p5]],
    "notes": {
        "environment": "local Spark notebook — not Dataproc JupyterHub",
        "hit_label": "average_rating >= 7.0 AND num_votes >= 1000",
    },
}

display(Markdown("### Core analytics — results summary"))
display(pd.DataFrame(
    [
        ("Job", metrics["job"]),
        ("Input data", metrics["input"]),
        ("Movies analyzed", metrics["row_count"]),
        ("Director prior vs rating correlation", round(metrics["director_prior_rating_corr"], 4)),
        ("High-rating / low-vote anomalies", metrics["anomaly_count_rating_ge_8_bottom_decile_votes"]),
        ("Base hit rate", None if metrics["base_hit_rate"] is None else round(metrics["base_hit_rate"], 4)),
    ],
    columns=["Result", "Value"],
))

display(Markdown("### Notes (same as saved in the metrics file)"))
display(pd.DataFrame(
    list(metrics["notes"].items()),
    columns=["Note", "Value"],
))

display(Markdown("### Charts shown above (also saved to disk)"))
display(pd.DataFrame(
    {
        "Chart": [
            "Genre × decade median rating",
            "Director prior vs film rating",
            "Runtime sweet spot",
            "Rating / vote anomalies",
            "Pre- vs post-release hit rate",
        ],
        "File": [p1.name, p2.name, p3.name, p4.name, p5.name],
    }
))


In [ ]:
# `metrics` (including notes) was built and displayed in the previous cell
out = metrics_dir / "analytics_metrics.json"
out.write_text(json.dumps(metrics, indent=2))
print("Saved metrics file for the report:", out)
spark.stop()